# 동물상 분류 모델 데이터
- 최소 데이터: 20개 클래스 20장씩


## 1. 데이터 수집

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, models, transforms
import numpy as np
import time

In [ ]:
from bing_image_downloader import downloader

downloader.download(
    query="알파카상",             # 검색어
    limit=50,
    output_dir="./dataset/raw",     # 파일 저장 위치
    adult_filter_off=True,
    force_replace=False,
    timeout=30 
)

downloader.download(
    query="공룡상",             # 검색어
    limit=50,
    output_dir="./dataset/raw",     # 파일 저장 위치
    adult_filter_off=True,
    force_replace=False,
    timeout=30 
)

downloader.download(
    query="원숭이상",             # 검색어
    limit=50,
    output_dir="./dataset/raw",     # 파일 저장 위치
    adult_filter_off=True,
    force_replace=False,
    timeout=30 
)

downloader.download(
    query="오리상",             # 검색어
    limit=50,
    output_dir="./dataset/raw",     # 파일 저장 위치
    adult_filter_off=True,
    force_replace=False,
    timeout=30 
)

downloader.download(
    query="코알라상",             # 검색어
    limit=50,
    output_dir="./dataset/raw",     # 파일 저장 위치
    adult_filter_off=True,
    force_replace=False,
    timeout=30 
)

downloader.download(
    query="늑대상",             # 검색어
    limit=50,
    output_dir="./dataset/raw",     # 파일 저장 위치
    adult_filter_off=True,
    force_replace=False,
    timeout=30 
)

downloader.download(
    query="햄스터상",             # 검색어
    limit=50,
    output_dir="./dataset/raw",     # 파일 저장 위치
    adult_filter_off=True,
    force_replace=False,
    timeout=30 
)

downloader.download(
    query="꽃돼지상",             # 검색어
    limit=50,
    output_dir="./dataset/raw",     # 파일 저장 위치
    adult_filter_off=True,
    force_replace=False,
    timeout=30 
)

downloader.download(
    query="너구리상",             # 검색어
    limit=50,
    output_dir="./dataset/raw",     # 파일 저장 위치
    adult_filter_off=True,
    force_replace=False,
    timeout=30 
)

downloader.download(
    query="말상",             # 검색어
    limit=50,
    output_dir="./dataset/raw",     # 파일 저장 위치
    adult_filter_off=True,
    force_replace=False,
    timeout=30 
)

## 2. train / test 데이터 분리 및 
- ArcFace 자동 그룹

- 특징: 얼굴 검출 + ArcFace 임베딩으로 “같은 사람 추정 그룹” 만들고, 그 그룹 단위로 train/tst 나눔.
- 장점: 사람 중복이 섞이는 걸 크게 줄여서 평가가 더 현실적.
- 부가: 얼굴 없는 이미지는 noface/로 따로 뺌.

In [12]:
import cv2, random, shutil, numpy as np
from pathlib import Path
from collections import defaultdict
from insightface.app import FaceAnalysis
from sklearn.cluster import DBSCAN

def imread_unicode(path: Path):
    data = np.fromfile(str(path), dtype=np.uint8)
    img = cv2.imdecode(data, cv2.IMREAD_COLOR)  # 항상 3채널
    return img

RAW_DIR = Path("dataset/raw")
OUT_DIR = Path("dataset")
SEED = 42
TRAIN_RATIO = 0.8
COPY_MODE = True
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

# DBSCAN 파라미터(같은 사람 묶는 강도)
# 엄격(덜 묶음): 0.30~0.33 / 기본: 0.35 / 느슨(더 묶음): 0.38~0.42
DBSCAN_EPS = 0.35
DBSCAN_MIN_SAMPLES = 2

SKIP_NO_FACE = True

def is_image(p: Path) -> bool:
    return p.suffix.lower() in IMG_EXTS

def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def copy_or_move(src: Path, dst: Path):
    if COPY_MODE:
        shutil.copy2(src, dst)
    else:
        shutil.move(str(src), str(dst))

def l2_normalize(x: np.ndarray) -> np.ndarray:
    n = np.linalg.norm(x)
    return x / (n + 1e-12)

random.seed(SEED)
np.random.seed(SEED)

# GPU 사용 (없으면 ctx_id=-1)
app = FaceAnalysis(name="buffalo_l")
app.prepare(ctx_id=0)

class_dirs = [d for d in RAW_DIR.iterdir() if d.is_dir()]
for split_name in ["train", "test"]:
    for cls in class_dirs:
        ensure_dir(OUT_DIR / split_name / cls.name)

noface_dir = OUT_DIR / "noface"
ensure_dir(noface_dir)

for cls in class_dirs:
    images = [p for p in cls.iterdir() if p.is_file() and is_image(p)]
    if not images:
        print(f"[WARN] no images: {cls.name}")
        continue

    # 1) 임베딩 추출
    paths, embs = [], []
    for p in images:
        img = imread_unicode(p)
        if img is None:
            raise RuntimeError(f"Failed to read image: {p}")

        faces = app.get(img)
        if len(faces) == 0:
            if SKIP_NO_FACE:
                copy_or_move(p, noface_dir / f"{cls.name}__{p.name}")
            continue

        # 가장 큰 얼굴 선택
        face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))
        emb = l2_normalize(face.embedding.astype(np.float32))

        paths.append(p)
        embs.append(emb)

    if len(paths) < 5:
        print(f"[WARN] too few usable faces: {cls.name}, usable={len(paths)}")
        continue

    X = np.vstack(embs)

    # 2) 같은 사람 추정 그룹핑(DBSCAN)
    clustering = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES, metric="cosine")
    labels = clustering.fit_predict(X)

    groups = defaultdict(list)
    singleton_id = 10**9
    for p, lb in zip(paths, labels):
        if lb == -1:
            groups[f"single_{singleton_id}"].append(p)
            singleton_id += 1
        else:
            groups[f"c{lb}"].append(p)

    group_list = list(groups.values())
    random.shuffle(group_list)

    # 3) 그룹 단위로 train 채우고 나머지 test
    total = sum(len(g) for g in group_list)
    train_target = int(total * TRAIN_RATIO)

    train_list, test_list = [], []
    train_count = 0
    for g in group_list:
        if train_count < train_target:
            train_list.extend(g)
            train_count += len(g)
        else:
            test_list.extend(g)

    # 4) 복사/이동
    for src in train_list:
        dst = OUT_DIR / "train" / cls.name / src.name
        copy_or_move(src, dst)

    for src in test_list:
        dst = OUT_DIR / "test" / cls.name / src.name
        copy_or_move(src, dst)

    print(f"{cls.name}: total={total} train={len(train_list)} test={len(test_list)} groups={len(group_list)}")

print("Done:", OUT_DIR.resolve())
print("noface:", noface_dir.resolve())

Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CUDAExecutionProvider': {'device_id': '0', 'has_user_compute_stream': '0', 'cudnn_conv1d_pad_to_nc1d': '0', 'user_compute_stream': '0', 'gpu_external_alloc': '0', 'gpu_mem_limit': '18446744073709551615', 'enable_cuda_graph': '0', 'gpu_external_free': '0', 'gpu_external_empty_cache': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'do_copy_in_default_stream': '1', 'cudnn_conv_use_max_workspace': '1', 'tunable_op_enable': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'prefer_nhwc': '0', 'use_ep_level_unified_stream': '0', 'use_tf32': '1', 'sdpa_kernel': '0', 'fuse_conv_bias': '0'}, 'CPUExecutionProvider': {}}
find model: C:\Users\user/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']

c:\workspace\project2\gachikium\.venv\Lib\site-packages\insightface\utils\face_align.py:23: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `SimilarityTransform.from_estimate` class constructor instead.
  tform.estimate(lmk, dst)


꽃돼지상: total=50 train=40 test=10 groups=31


c:\workspace\project2\gachikium\.venv\Lib\site-packages\insightface\utils\face_align.py:23: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `SimilarityTransform.from_estimate` class constructor instead.
  tform.estimate(lmk, dst)


너구리상: total=50 train=40 test=10 groups=30


...

원숭이상: total=50 train=41 test=9 groups=27


c:\workspace\project2\gachikium\.venv\Lib\site-packages\insightface\utils\face_align.py:23: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `SimilarityTransform.from_estimate` class constructor instead.
  tform.estimate(lmk, dst)


코알라상: total=50 train=41 test=9 groups=29


c:\workspace\project2\gachikium\.venv\Lib\site-packages\insightface\utils\face_align.py:23: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `SimilarityTransform.from_estimate` class constructor instead.
  tform.estimate(lmk, dst)


햄스터상: total=50 train=41 test=9 groups=20
Done: C:\workspace\project2\gachikium\유현희\dataset
noface: C:\workspace\project2\gachikium\유현희\dataset\noface


## 3. 얼굴 추출

In [ ]:
import cv2, numpy as np
from pathlib import Path
from insightface.app import FaceAnalysis
from insightface.utils.face_align import norm_crop

# ---------- 유니코드(한글 경로)에서도 안전한 이미지 읽기/쓰기 ----------
def imread_unicode(path: Path):
    data = np.fromfile(str(path), dtype=np.uint8)
    return cv2.imdecode(data, cv2.IMREAD_COLOR)

def imwrite_unicode(path: Path, img):
    path.parent.mkdir(parents=True, exist_ok=True)
    ext = path.suffix.lower()
    if ext not in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
        path = path.with_suffix(".jpg")
        ext = ".jpg"
    ok, buf = cv2.imencode(ext, img)
    if not ok:
        raise RuntimeError(f"imencode failed: {path}")
    buf.tofile(str(path))

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

# ---------- fallback: 중앙 정사각 크롭 -> 112 resize ----------
def center_square_resize(img, size=112):
    h, w = img.shape[:2]
    s = min(h, w)
    y1 = (h - s) // 2
    x1 = (w - s) // 2
    crop = img[y1:y1+s, x1:x1+s]
    return cv2.resize(crop, (size, size), interpolation=cv2.INTER_AREA)

# ---------- 얼굴 검출 재시도 ----------
def detect_best_face(app, img):
    """
    여러 조건으로 얼굴 검출을 재시도하고,
    잡히면 (face, used_img, padded_offset) 반환.
    - padded_offset: padding을 줬을 경우 원본 좌표계 보정용(여기선 align만 쓰니 크게 필요없지만 남겨둠)
    """
    # 1) 원본 그대로
    faces = app.get(img)
    if faces:
        face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))
        return face, img, (0,0)

    # 2) padding 후 재시도
    h, w = img.shape[:2]
    pad = int(0.25 * max(h, w))
    img_pad = cv2.copyMakeBorder(img, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=(0,0,0))
    faces = app.get(img_pad)
    if faces:
        face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))
        return face, img_pad, (pad, pad)

    # 3) 업스케일 후 재시도(작은 얼굴 대비)
    scale = 1.5
    img_up = cv2.resize(img, (int(w*scale), int(h*scale)), interpolation=cv2.INTER_CUBIC)
    faces = app.get(img_up)
    if faces:
        face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))
        return face, img_up, (0,0)

    return None, img, (0,0)

# ---------- 설정 ----------
INPUTS = [
    (Path("dataset/train"), Path("dataset/face_train")),
    (Path("dataset/test"),  Path("dataset/face_test")),
]

# ✅ 검출을 “좀 더 잘 잡게” 준비 (임계값 낮추고 det_size 키움)
app = FaceAnalysis(name="buffalo_l")
app.prepare(ctx_id=0, det_size=(1280, 1280), det_thresh=0.1)

total = 0
aligned_ok = 0
fallback_cnt = 0

for in_root, out_root in INPUTS:
    for cls_dir in [d for d in in_root.iterdir() if d.is_dir()]:
        imgs = [p for p in cls_dir.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS]
        for p in imgs:
            total += 1
            img = imread_unicode(p)
            if img is None:
                # 읽기 실패도 fallback으로 “무조건” 생성
                # (원본 파일이 실제로 깨진 경우라면 이조차 안 될 수 있음)
                continue

            face, used_img, _ = detect_best_face(app, img)

            if face is not None and hasattr(face, "kps") and face.kps is not None:
                # ✅ 가장 안정적인 방식: 랜드마크 기반 정렬 112x112
                out = norm_crop(used_img, face.kps)  # (112,112,3)
                out_name = p.with_suffix(".jpg").name
                imwrite_unicode(out_root / cls_dir.name / out_name, out)
                aligned_ok += 1
            else:
                # ✅ 검출 실패해도 “무조건” 결과 생성: 중앙 크롭 fallback
                out = center_square_resize(img, size=112)
                out_name = p.with_suffix(".jpg").stem + "__FALLBACK.jpg"
                imwrite_unicode(out_root / cls_dir.name / out_name, out)
                fallback_cnt += 1

print(f"Done. total={total}, aligned={aligned_ok}, fallback={fallback_cnt}")
print("Outputs:", Path("dataset/face_train").resolve(), Path("dataset/face_test").resolve())

Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CUDAExecutionProvider': {'device_id': '0', 'has_user_compute_stream': '0', 'cudnn_conv1d_pad_to_nc1d': '0', 'user_compute_stream': '0', 'gpu_external_alloc': '0', 'gpu_mem_limit': '18446744073709551615', 'enable_cuda_graph': '0', 'gpu_external_free': '0', 'gpu_external_empty_cache': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'do_copy_in_default_stream': '1', 'cudnn_conv_use_max_workspace': '1', 'tunable_op_enable': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'prefer_nhwc': '0', 'use_ep_level_unified_stream': '0', 'use_tf32': '1', 'sdpa_kernel': '0', 'fuse_conv_bias': '0'}, 'CPUExecutionProvider': {}}
find model: C:\Users\user/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']

c:\workspace\project2\gachikium\.venv\Lib\site-packages\insightface\utils\face_align.py:23: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `SimilarityTransform.from_estimate` class constructor instead.
  tform.estimate(lmk, dst)


Done. total=500, aligned=483, fallback=17
Outputs: C:\workspace\project2\gachikium\유현희\dataset\face_train_112 C:\workspace\project2\gachikium\유현희\dataset\face_test_112
